![CCAI 9012 course banner](../../../docs/figs/brand_identity/banner.png)

# LLM System Prompt Playground

## Same question, different assistant

Suppose three assistants receive exactly the same neighbourhood question. One answers as an urban planning advisor, one as a beginner-friendly tutor, and one as a critical reviewer. Their answers may emphasize different things even though the user input is unchanged.

In this lab, you will run that controlled comparison yourself.


## What you will learn

By the end of this notebook, you should be able to:

- distinguish a **system prompt** from a **user message**;
- explain how prompting can shape an LLM's tone, focus, and response structure;
- compare roles fairly by changing one part of the input at a time;
- trace the simple application flow **input → model → output**.


## Roadmap

| Step | What you will do |
|---|---|
| 1 | Recap what an LLM receives and returns. |
| 2 | Separate the system role from the user's task. |
| 3 | Set up the course model client. |
| 4 | Run one role, then compare three roles. |
| 5 | Design and test your own role prompt. |


## Step 1 — Recap: what goes into an LLM application?

At the simplest level, this is a **text-to-text** system:

<img src="llm_prompt_comparison_flow.svg" alt="A controlled LLM prompt experiment that holds the user question fixed while changing the system role, then compares responses" width="100%">

*Figure 1. A controlled system-prompt comparison.*

The **system prompt** gives reusable role and style guidance, while the **user message** supplies today's changing task. Each click sends a fresh pair of messages. The diagram also reminds us that a system prompt guides behaviour but does not, by itself, guarantee a factually correct response.


## Step 2 — Separate the role from the task

Read this pair:

**System prompt**  
`You are a plain-language tutor. Explain ideas for a beginner using everyday language and a short example.`

**User message**  
`How can a dense neighbourhood reduce summer heat?`

The system prompt says **how to respond**. The user message says **what to respond to**.

A reusable system prompt often includes:

- **Role** — Who should the assistant act like?
- **Audience and tone** — Who is the answer for, and how should it sound?
- **Focus** — What should the assistant pay attention to?
- **Response shape** — Should it use steps, bullets, examples, or another format?

Avoid putting the specific heat question into a reusable role prompt. We want to swap roles while keeping the task unchanged.

### Quick check

Where would each instruction belong?

1. “You are a critical reviewer who checks assumptions.”
2. “List three ways this neighbourhood can add shade.”
3. “Write for a community group with no technical background.”

Suggested answer: 1 and 3 are stable system guidance; 2 is the changing user task.


## Step 3 — Prepare the model connection

Run the next two cells in order.

The first imports the interface and course helpers. The second creates one reusable DeepSeek client. The course environment already includes `ipywidgets`, `langchain-core`, `langchain-deepseek`, and `ccai9012`.

Your API key must be configured locally in `ccai9012/token.yaml` or `DEEPSEEK_API_KEY`. The notebook does not display or store the key in its output.


In [1]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
from langchain_core.messages import HumanMessage, SystemMessage

from ccai9012.llm_utils import get_deepseek_api_key, initialize_llm

In [4]:
# This setup cell creates one reusable client.
llm = None
setup_error = None

try:
    api_key = get_deepseek_api_key()
    if not api_key:
        raise RuntimeError("No DeepSeek API key was found.")
    llm = initialize_llm(
        temperature=0.4,
        max_tokens=None,  # No notebook-level response-length cap.
        api_key=api_key,
    )
    print("Model client is ready. Run the interface cell below.")
except Exception:
    setup_error = (
        "The model client could not be initialized. Configure DEEPSEEK_API_KEY "
        "in ccai9012/token.yaml or your environment, then rerun this cell."
    )
    print(setup_error)

Model client is ready. Run the interface cell below.


## Step 4 — Run a controlled prompt experiment

Start with **Urban Planning Advisor** and click **Run model**. Then click **Compare three roles**.

For a fair comparison:

1. Keep the user question unchanged.
2. Read all three responses once for the main idea.
3. Read them again and look for differences in tone, priorities, detail, and structure.
4. Do not assume a confident answer is automatically correct.

The comparison makes three requests with the same user message. The designed change is the system prompt.


In [5]:
ROLE_PROMPTS = {
    "Urban Planning Advisor": (
        "You are an urban planning advisor. Give practical, actionable recommendations "
        "and organize your answer as concise actions."
    ),
    "Plain-Language Tutor": (
        "You are a plain-language tutor. Explain ideas for a beginner using clear, "
        "everyday language and a short example when useful."
    ),
    "Critical Reviewer": (
        "You are a critical reviewer. Emphasize benefits, trade-offs, assumptions, "
        "and questions to check before acting."
    ),
}

playground_title = widgets.HTML(
    value="<h2 style='margin: 0 0 12px 0;'>LLM Prompt Playground</h2>"
)

role_dropdown = widgets.Dropdown(
    options=[*ROLE_PROMPTS, "Custom"],
    value="Urban Planning Advisor",
    description="",
    layout=widgets.Layout(width="100%"),
)
system_prompt = widgets.Textarea(
    value=ROLE_PROMPTS["Urban Planning Advisor"],
    description="",
    placeholder="Describe the role and instructions for the model.",
    layout=widgets.Layout(width="100%", height="110px"),
)
user_input = widgets.Textarea(
    value="How can a dense neighbourhood reduce summer heat?",
    description="",
    layout=widgets.Layout(width="100%", height="90px"),
)
run_button = widgets.Button(description="Run model", button_style="primary", icon="play")
compare_button = widgets.Button(description="Compare three roles", icon="columns")
response_panel = widgets.Output(layout=widgets.Layout(border="1px solid #c7c7c7", padding="12px"))

def update_system_prompt(change):
    """Load a supplied role prompt; Custom deliberately preserves the current text."""
    selected_role = change["new"]
    if selected_role in ROLE_PROMPTS:
        system_prompt.value = ROLE_PROMPTS[selected_role]

def get_response_text(response):
    """Return displayable text from a LangChain chat response."""
    content = response.content
    return content if isinstance(content, str) else str(content)

def ask_with_role(role_prompt, question):
    """Send genuine system and user chat messages through the shared client."""
    messages = [SystemMessage(content=role_prompt), HumanMessage(content=question)]
    return get_response_text(llm.invoke(messages))

def validate_inputs(require_system=True):
    if llm is None:
        return setup_error or "Model client is unavailable. Run the setup cell after configuring credentials."
    if require_system and not system_prompt.value.strip():
        return "Enter a System Prompt before running the model."
    if not user_input.value.strip():
        return "Enter a User Input before making a request."
    return None

def set_generating(is_generating):
    run_button.disabled = is_generating
    compare_button.disabled = is_generating

def show_message(message):
    with response_panel:
        clear_output(wait=True)
        display(Markdown(message))

def show_comparisons(comparisons):
    """Display each response in a clearly separated comparison section."""
    with response_panel:
        clear_output(wait=True)
        display(Markdown("## Same user input, three roles"))
        for role_name, answer in comparisons:
            display(widgets.HTML(value=(
                f"<h3 style='margin: 22px 0 8px; padding: 8px 10px; "
                f"background: #f2f5f7; border-left: 4px solid #4a90d9;'>"
                f"{role_name}</h3>"
            )))
            display(Markdown(answer))
            display(widgets.HTML(value="<hr style='margin: 22px 0; border: 0; border-top: 2px solid #d9e1e8;'>"))

def run_model(_):
    problem = validate_inputs()
    if problem:
        show_message(f"**Ready to fix:** {problem}")
        return
    set_generating(True)
    try:
        show_message("_Generating a response…_")
        answer = ask_with_role(system_prompt.value.strip(), user_input.value.strip())
        show_message(f"### Response\n\n{answer}")
    except Exception:
        show_message("**Request failed.** Check your network connection, API access, and local credential configuration, then try again.")
    finally:
        set_generating(False)

def compare_roles(_):
    problem = validate_inputs(require_system=False)
    if problem:
        show_message(f"**Ready to fix:** {problem}")
        return
    set_generating(True)
    try:
        show_message("_Generating three role-specific responses…_")
        question = user_input.value.strip()
        comparisons = []
        for role_name, role_prompt in ROLE_PROMPTS.items():
            comparisons.append((role_name, ask_with_role(role_prompt, question)))
        show_comparisons(comparisons)
    except Exception:
        show_message("**Comparison failed.** Check your network connection, API access, and local credential configuration, then try again.")
    finally:
        set_generating(False)

role_dropdown.observe(update_system_prompt, names="value")
run_button.on_click(run_model)
compare_button.on_click(compare_roles)

display(widgets.VBox([
    playground_title,
    widgets.HTML("<b>Role preset</b>"), role_dropdown,
    widgets.HTML("<b>System Prompt</b>"), system_prompt,
    widgets.HTML("<b>User Input</b>"), user_input,
    widgets.HBox([run_button, compare_button]), response_panel,
]))
show_message("Edit the prompts, then run one role or compare all three.")

### Record what you observe

| Role | Tone or audience | What does it emphasize? | One useful feature | One limitation |
|---|---|---|---|---|
| Urban Planning Advisor |  |  |  |  |
| Plain-Language Tutor |  |  |  |  |
| Critical Reviewer |  |  |  |  |

Look for evidence in the actual wording. For example, did a response use action verbs, explain a term, or question an assumption?


## Step 5 — Design your own system role

Select **Custom** and write a system prompt for a neighbourhood-association funding advisor.

Use this scaffold:

> You are a **[role]**. Write for **[audience]** in a **[tone]**. Focus on **[priority]**. Organize the response as **[shape]**.

Come up with your own prompt in **User Input**.

1. Run your custom role.
2. Compare it with the Critical Reviewer.
3. Identify one change that came from your system prompt.
4. Revise one phrase and run it again. Did the response move in the direction you expected?


## Reflection and wrap-up

- Which response changed most clearly because of its role?
- Which part changed: tone, detail, structure, priorities, or something else?
- Why does keeping the user message fixed make this a stronger experiment?
- When might a strong role prompt still produce an unreliable answer?

### Three takeaways

1. An LLM application can combine a stable system role with a changing user task.
2. Prompt wording shapes behaviour, but it does not guarantee factual accuracy.
3. A good comparison changes one variable at a time and checks the resulting evidence.

Model responses can vary between runs. Verify important claims with reliable sources, especially when the output may influence real decisions.

For foundations, see [LLM basics](../week2/week2_t_llm_basics.ipynb). For a later lesson, see [LLM output control](../week8/week8_ic_llm_output_control.ipynb).
